# Datos Masivos — McCain: ¿dónde invertir para vender más papas a la francesa?

**Índice de inversión por zona metropolitana, en Databricks**

McCain no compra papa mexicana: importa papa congelada (variedades como Russet) y la vende a
restaurantes. La pregunta que sí le sirve es **dónde está creciendo la demanda**:

> Si McCain tuviera X millones para crecer en México, ¿en qué zonas metropolitanas los pondría?

Para cada una de las **92 zonas metropolitanas de CONAPO** calculamos cuatro componentes con el
DENUE de INEGI (2019 contra 2026), los convertimos en percentiles de 0 a 100 y los promediamos
con el **mismo peso** (25% cada uno):

| Componente | Qué mide | Fuente |
|---|---|---|
| A. Crecimiento | tasa anual de restaurantes donde se venden papas (comida rápida, autoservicio, a la carta) | DENUE |
| B. Cadenas | aperturas netas de McDonald's, KFC, Carl's Jr., Burger King… por 100 mil habitantes | DENUE |
| Turismo | hoteles por 100 mil habitantes | DENUE |
| Mercado | número de restaurantes objetivo hoy | DENUE |
| C. Competencia | de qué país vienen las papas importadas y a qué precio por kilo (nacional) | UN Comtrade |

Igual que el notebook 03, aquí no copiamos código: llamamos a los módulos de `src/` del
repositorio. **Cómo correrlo:** conectar a *Serverless* y dar **Run all** (unos 15 minutos la
primera vez, por la descarga del DENUE).

## 0. Instalar librerías

In [ ]:
%pip install -q beautifulsoup4 pyyaml

In [ ]:
dbutils.library.restartPython()

## 1. Configurar el data lake

Mismo Volume que el notebook 03. Las fuentes nuevas quedan en sus propias carpetas:
`raw/denue/<versión>/`, `raw/metropolis/` y `raw/comtrade/`.

In [ ]:
import os
import sys
from pathlib import Path

CATALOGO = "workspace"
ESQUEMA = "mccain_papa"
VOLUMEN = "datalake"
LAGO = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

os.environ["DATOS_MASIVOS_DATA"] = LAGO
os.environ["DATOS_MASIVOS_REPORTS"] = f"{LAGO}/reports"


def encontrar_raiz():
    """Buscamos hacia arriba la carpeta del repo (la que contiene src/)."""
    for candidata in [Path.cwd(), *Path.cwd().parents]:
        if (candidata / "src" / "processing" / "build_dataset.py").exists():
            return candidata
    raise RuntimeError("Abre este notebook desde el Git folder del repositorio.")


RAIZ = encontrar_raiz()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

from src.utils import rutas
from src.utils.config import cargar_config

CONFIG = cargar_config()
print("Data lake :", rutas.DATA)
print("Versiones del DENUE:", [v["etiqueta"] for v in CONFIG["denue"]["versiones"]])

## 2. Adquisición

Tres fuentes, todas abiertas y sin llave:

| Fuente | Qué bajamos | Tamaño |
|---|---|---|
| DENUE (INEGI) | sector 72, 4 versiones: 2015, 2019, 2022 y 2026 (7 zip) | ~300 MB en zip, ~1.5 GB en CSV |
| CONAPO | catálogo de 92 metrópolis y sus municipios | < 1 MB |
| UN Comtrade | importaciones mensuales de papa congelada (HS 200410), 2015–2025 | 132 consultas |

Si Databricks no alcanza INEGI, el plan B es el mismo que en el notebook 03: en local corremos
`python -m src.ingestion.download_data --fuente indice`, comprimimos `data/raw` como
`raw_indice.zip` y lo subimos al Volume.

In [ ]:
import requests

PORTALES = {
    "INEGI": "https://www.inegi.org.mx/app/descarga/",
    "Comtrade": "https://comtradeapi.un.org/public/v1/preview/C/M/HS?reporterCode=484&period=202401&cmdCode=200410&flowCode=M",
}


def responde(url):
    try:
        return requests.get(url, timeout=20).status_code < 500
    except requests.RequestException:
        return False


alcance = {fuente: responde(url) for fuente, url in PORTALES.items()}
ZIP = Path(LAGO) / "raw_indice.zip"
MODO = "completo" if all(alcance.values()) else ("zip" if ZIP.exists() else "sin_red")
print("Portales:", alcance, "-> modo", MODO)

In [ ]:
import zipfile

from src.ingestion import download_data

if MODO == "completo":
    # Lo que ya bajó se toma de la caché: correr esta celda otra vez no repite descargas.
    codigo = download_data.main(["--fuente", "indice"])
elif MODO == "zip":
    with zipfile.ZipFile(ZIP) as archivo:
        archivo.extractall(LAGO)
    codigo = 0
else:
    raise RuntimeError("Databricks no alcanza INEGI/Comtrade y no hay raw_indice.zip en el Volume")
print("Código de salida:", codigo)

for carpeta in ["denue", "metropolis", "comtrade"]:
    archivos = [a for a in (rutas.RAW / carpeta).rglob("*") if a.is_file()]
    mb = sum(a.stat().st_size for a in archivos) / 1024**2
    print(f"raw/{carpeta:10s} | {len(archivos):4d} archivos | {mb:7.1f} MB")

## 3. Procesamiento con Spark

`build_dataset --fuente indice` extrae los CSV de los zip, lee cada versión del DENUE con su
propia codificación (2015 viene en UTF-8 con encabezados largos; 2019 en adelante en Latin-1 con
encabezados cortos), las une con `unionByName` y escribe `interim/` y `processed/` particionados
por versión.

In [ ]:
import json

from src.processing import build_dataset

codigo = build_dataset.main(["--fuente", "indice"])
print("Código de salida:", codigo)
resumen = json.loads((rutas.PROCESSED / "resumen_build.json").read_text())
resumen["indice"]

In [ ]:
from pyspark.sql import functions as F

denue = spark.read.parquet(str(rutas.PROCESSED / "denue_establecimientos"))
display(
    denue.groupBy("version")
    .pivot("categoria")
    .count()
    .orderBy("version")
)

## 4. Índice de inversión

Cruzamos el DENUE con el catálogo de municipios de CONAPO (*broadcast join*: el catálogo tiene
~420 filas), contamos por metrópoli y versión, y calculamos los percentiles con una *window*.

In [ ]:
from src.analysis import indice_inversion

codigo = indice_inversion.main()
print("Código de salida:", codigo)

## 5. Validación

In [ ]:
import pandas as pd

from src.validation import validate_dataset

codigo = validate_dataset.main()
validacion = pd.read_csv(rutas.TABLAS / "validacion.csv")
print("Código de salida:", codigo, validacion.estado.value_counts().to_dict())
display(validacion[validacion.tabla.isin(["denue", "metropolis", "metropolis_municipios",
                                          "importaciones", "indice"])])

## 6. Gráficas

Las mismas tres gráficas que genera el pipeline en `reports/figures/`, una por pregunta.

In [ ]:
from src.analysis import graficas

figuras = graficas.generar_todas(spark)

### ¿Dónde invertir?
Cada barra se parte en lo que aporta cada componente al índice.

In [ ]:
display(figuras["ranking_metropolis"])

### ¿Por qué ahí?
Arriba a la izquierda: crece rápido y todavía hay pocos restaurantes por habitante.

In [ ]:
display(figuras["mapa_oportunidad"])

### ¿Contra quién competimos?
De dónde vienen las papas congeladas que importa México y a qué precio por kilo.

In [ ]:
if "competencia_importacion" in figuras:
    display(figuras["competencia_importacion"])

## 7. Tablas en el catálogo

Registramos los resultados como tablas Delta en `workspace.mccain_papa`.

In [ ]:
TABLAS_CATALOGO = {
    "indice_inversion": "Indice de inversion por zona metropolitana (0-100) y sus componentes",
    "serie_metropolis": "Restaurantes objetivo, cadenas y hoteles por metropoli y version del DENUE",
    "cadenas_metropolis": "Sucursales por cadena, metropoli y version del DENUE",
    "metropolis": "Catalogo CONAPO 2020: poblacion y PIB por metropoli",
    "importaciones": "Importaciones mensuales de papa congelada por pais (UN Comtrade, HS 200410)",
    "importaciones_anuales": "Importaciones anuales por pais: valor, kilos, US$/kg y participacion",
    "denue_establecimientos": "DENUE sector 72, todas las versiones, con categoria y cadena",
}

for tabla, descripcion in TABLAS_CATALOGO.items():
    ruta = rutas.PROCESSED / tabla
    if not ruta.exists():
        print(f"(sin {tabla})")
        continue
    nombre = f"{CATALOGO}.{ESQUEMA}.{tabla}"
    (spark.read.parquet(str(ruta)).write.mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(nombre))
    spark.sql(f"COMMENT ON TABLE {nombre} IS '{descripcion}'")
    print(f"{nombre:50s} {spark.table(nombre).count():>10,} filas")

## 8. Resultado: top 10 de zonas metropolitanas

In [ ]:
display(spark.sql(f"""
    SELECT ranking, metropoli,
           indice,
           ROUND(crecimiento_anual, 1)  AS crecimiento_pct_anual,
           ROUND(cadenas_netas_100k, 1) AS cadenas_netas_100k,
           ROUND(hoteles_100k, 1)       AS hoteles_100k,
           objetivo_final               AS restaurantes_objetivo,
           poblacion_2020
    FROM {CATALOGO}.{ESQUEMA}.indice_inversion
    ORDER BY ranking
    LIMIT 10
"""))

**Para verla como gráfica de Databricks** (además de las de arriba):

1. En la tabla de resultados de la celda anterior, dar clic en **+** → **Visualization**.
2. *Visualization type*: **Bar**.
3. *X column*: `metropoli`. *Y columns*: `indice` (agregación **Sum**).
4. En la pestaña **X axis**, *Sort values*: desactivado, para respetar el orden del ranking.
5. **Save**. La gráfica queda guardada en el notebook, debajo de la tabla.

In [ ]:
display(spark.sql(f"""
    SELECT anio, pais, ROUND(valor_usd / 1e6, 1) AS millones_usd,
           ROUND(usd_kg, 2) AS usd_kg, ROUND(participacion, 1) AS participacion_pct
    FROM {CATALOGO}.{ESQUEMA}.importaciones_anuales
    WHERE participacion >= 1
    ORDER BY anio DESC, valor_usd DESC
"""))

## 9. Partition pruning sobre el DENUE

`denue_establecimientos` está particionada por `version`. Al pedir solo 2026, Spark lee una de
las cuatro carpetas: buscamos `PartitionFilters` en el plan.

In [ ]:
(spark.read.parquet(str(rutas.PROCESSED / "denue_establecimientos"))
 .filter(F.col("version") == "2026_05")
 .groupBy("cadena").count()
 .explain(mode="formatted"))

## 10. Conclusión

1. **La pregunta cambió a una que McCain puede usar:** dónde crece la demanda de papa a la
   francesa, no dónde se siembra papa.
2. **El índice ordena las 92 metrópolis** con cuatro componentes de peso igual; la tabla
   `indice_inversion` dice cuál pesa más en cada una.
3. **La competencia es nacional:** Comtrade muestra de qué país vienen las papas importadas y a
   qué precio por kilo.
4. **Mismo pipeline local y en Databricks**, con el data lake en el Volume y las tablas en
   Unity Catalog.